In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_white, acorr_breusch_godfrey
from scipy.stats import jarque_bera
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. CARREGAMENTO E PREPARAÇÃO DOS DADOS ---
# (Repetindo a unificação para garantir consistência neste novo notebook)

try:
    pib_df = pd.read_pickle('pib_tratado.pkl')
    desemprego_df = pd.read_pickle('desemprego_tratado.pkl')

    # Padronizar nome da coluna se necessário
    if 'desemprego' in desemprego_df.columns:
        desemprego_df.rename(columns={'desemprego': 'u_t'}, inplace=True)

    # Unificar
    df_model = pib_df.join(desemprego_df, how='inner')

    # Filtrar período (conforme definido na análise anterior)
    df_model = df_model[df_model.index <= '2024-12-31'].copy()

    # Criar variáveis do Modelo de Primeira Diferença
    # Δu_t = u_t - u_{t-1}
    df_model['delta_u_t'] = df_model['u_t'].diff()
    
    # Crescimento PIB já existe, mas garantindo o cálculo:
    df_model['crescimento_pib'] = df_model['ln_pib'].diff() * 100

    # Remover NaNs gerados pelas diferenças
    df_model.dropna(inplace=True)

    print(f"Dados carregados. Período: {df_model.index.min().date()} a {df_model.index.max().date()}")
    print(f"Nº de observações: {len(df_model)}")

except FileNotFoundError:
    print("Erro: Arquivos .pkl não encontrados. Execute os notebooks anteriores primeiro.")


# --- 2. ESTIMAÇÃO DO MODELO OLS ---
# Equação: Δu_t = β0 + β1 * crescimento_pib_t + ε_t

# Definir variáveis dependente (Y) e independente (X)
Y = df_model['delta_u_t']
X = df_model['crescimento_pib']

# Adicionar constante (β0)
X = sm.add_constant(X)

# Ajustar o modelo
modelo_ols = sm.OLS(Y, X).fit()

# Exibir resultados
print("\n" + "="*60)
print("RESULTADOS DO MODELO 1: PRIMEIRA DIFERENÇA (OLS)")
print("="*60)
print(modelo_ols.summary())


# --- 3. DIAGNÓSTICO DOS RESÍDUOS (Fase 2 da Documentação) ---
residuos = modelo_ols.resid

print("\n" + "="*60)
print("TESTES DE DIAGNÓSTICO")
print("="*60)

# A. Autocorrelação Serial (Breusch-Godfrey)
# H0: Não há autocorrelação serial até a defasagem p
bg_test = acorr_breusch_godfrey(modelo_ols, nlags=4)
print(f"1. Autocorrelação (Breusch-Godfrey, lags=4):")
print(f"   - Estatística LM: {bg_test[0]:.4f}")
print(f"   - P-valor: {bg_test[1]:.4f}")
if bg_test[1] < 0.05:
    print("   -> Rejeita H0: Há indícios de autocorrelação serial.")
else:
    print("   -> Não rejeita H0: Não há indícios fortes de autocorrelação.")

# B. Heterocedasticidade (White)
# H0: A variância dos erros é constante (Homocedasticidade)
white_test = het_white(residuos, modelo_ols.model.exog)
print(f"\n2. Heterocedasticidade (Teste de White):")
print(f"   - Estatística LM: {white_test[0]:.4f}")
print(f"   - P-valor: {white_test[1]:.4f}")
if white_test[1] < 0.05:
    print("   -> Rejeita H0: Há indícios de heterocedasticidade.")
else:
    print("   -> Não rejeita H0: Resíduos parecem homocedásticos.")

# C. Normalidade (Jarque-Bera)
# H0: Os resíduos seguem uma distribuição normal
jb_stat, jb_p = jarque_bera(residuos)
print(f"\n3. Normalidade (Jarque-Bera):")
print(f"   - Estatística JB: {jb_stat:.4f}")
print(f"   - P-valor: {jb_p:.4f}")
if jb_p < 0.05:
    print("   -> Rejeita H0: Resíduos não seguem distribuição normal.")
else:
    print("   -> Não rejeita H0: Resíduos parecem normais.")


# --- 4. CORREÇÃO HAC (Se necessário) ---
# Se houver autocorrelação ou heterocedasticidade, usamos erros padrão robustos (Newey-West)
print("\n" + "="*60)
print("ESTIMAÇÃO ROBUSTA (HAC - Newey-West)")
print("="*60)
# maxlags geralmente T^(1/4) ~ 3 ou 4 para dados trimestrais
modelo_hac = sm.OLS(Y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
print(modelo_hac.summary())


# --- 5. VISUALIZAÇÃO DOS RESULTADOS ---
plt.figure(figsize=(12, 6))

# Gráfico de Dispersão com Linha de Regressão
sns.regplot(x=df_model['crescimento_pib'], y=df_model['delta_u_t'], 
            scatter_kws={'alpha':0.6, 'color':'blue'}, line_kws={'color':'red'})

plt.title('Lei de Okun (Brasil): Variação do Desemprego vs Crescimento do PIB', fontsize=14)
plt.xlabel('Crescimento do PIB (%)')
plt.ylabel('Variação do Desemprego (p.p.)')
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.grid(True, alpha=0.3)

# Adicionar a equação no gráfico
beta0 = modelo_hac.params['const']
beta1 = modelo_hac.params['crescimento_pib']
texto_eq = f'Δu = {beta0:.2f} {beta1:.2f} * Δy'
plt.annotate(texto_eq, xy=(0.05, 0.9), xycoords='axes fraction', fontsize=12, 
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8))

plt.show()
